# Stage 2: Lin-KK Quality

Validates each spectrum by Kramers-Kronig compliance and picks the best replica per (condition, T).

**Reads:** `{sample_id}/Results/{condition}/stage1_labeling.xlsx` · `{sample_id}/ISM validation/*.ism`
**Writes:** `{sample_id}/Results/{condition}/stage2_kk.xlsx`

Set `FOCUS_T` to process one temperature at a time. The export is merge-aware: other temperatures are preserved in `stage2_kk.xlsx`.

## Quick links
- [Configuration](#configuration): sample_id, condition_filter, FOCUS_T, KK parameters
- [KK_OVERRIDES](#configuration): per-(condition, T) frequency cutoffs
- [OVERRIDES](#configuration): manual replica selection
- [Step 2](#step-2-review-flagged-spectra): review flagged spectra, tune frequency cutoffs
- [Step 3](#step-3-selection-summary-and-export): selection summary and export

## Configuration

In [ ]:
import json
from pathlib import Path

_cfg      = json.loads(Path("session.json").read_text()) if Path("session.json").exists() else {}
sample_id = _cfg.get("sample_id") or input("Sample folder name: ").strip()

# conditions saved in stage 0; leave empty to process all
_saved          = _cfg.get("conditions", [])
condition_filter = _saved  # override here if needed, e.g. condition_filter = ["cond_A"]

FOCUS_CONDITION = None   # process only this condition (None = all)
FOCUS_T         = None   # process only this temperature (None = all, e.g. 600)
SKIP_EXISTING   = False  # skip conditions that already have stage2_kk.xlsx

# Lin-KK: M selection
KK_USE_BINARY_M = False  # False = linear search (reproducible, RelaxIS default)
                          # True  = binary search (faster but non-deterministic on noisy spectra)
KK_MU_TARGET    = 0.50   # sign-change fraction target (RelaxIS default)
KK_C            = 0.76   # M = round(KK_C x N); used only when KK_USE_BINARY_M = False

# Adaptive IQR cutoffs
KK_IQR_FENCE  = 0.5  # IQR fence multiplier; lower = tighter cut at noisy edges
KK_IQR_WINDOW = 10   # consecutive clean points required to confirm cut edge

# Hard frequency limits (global defaults); both None = use adaptive IQR only
KK_F_MIN_HARD = 50    # [Hz] remove data below this frequency (LF electrode noise)
KK_F_MAX_HARD = None  # [Hz] None lets adaptive IQR pick

# False = standard kk_score >= 0.97 (publication default, matches Schonleber 2014)
# True  = ceramic-aware dual criterion: W_re >= 0.95 AND W_im >= 0.93
#         useful when high-resistance spectra show W_im systematically 0.02-0.04
#         below W_re due to the small modulus of Z_im at these temperatures
KK_USE_W_CRITERIA = False

# KK_OVERRIDES: per-(condition, T) frequency cutoff tuning
# Use when Step 1 produces YELLOW/RED spectra; Cell B auto-suggests values.
# Priority: temperature-specific > condition-level > global KK_F_MIN/MAX_HARD
#
# Example (single condition, different limits per temperature):
# KK_OVERRIDES = {
#     "sample_Gas-SCCM_Tmax_Tmin_delta": {
#         "f_min_hard": 30.0,           # condition-level default
#         550: {"f_min_hard": 40.0, "f_max_hard": 1e6},
#         600: {"f_min_hard": 50.0, "f_max_hard": 5e5},
#     }
# }
KK_OVERRIDES = {}

# OVERRIDES: manual replica selection
# Force a specific file (string) or compare a list and auto-select best kk_score (list).
#
# Example:
# OVERRIDES = {
#     "sample_Gas-SCCM_Tmax_Tmin_delta": {
#         550: "filename_550C.ism",                          # force one file
#         400: ["filename_400C_1.ism", "filename_400C_2.ism"],  # compare and pick best
#     }
# }
OVERRIDES = {}

### FOCUS toggle — process a single temperature

Click **FOCUS** to restrict processing to one temperature (handy when batch-processing many
files and you want to fine-tune a single T). Pick the T, then **re-run the batch cell**. Toggle
off — or re-run the config cell — to process all temperatures again.

In [ ]:
# FOCUS selector — pick a condition and/or temperature by clicking (no config edit).
import sys as _sys
from pathlib import Path as _Path
_nbdir = _Path().resolve()
if str(_nbdir) not in _sys.path:
    _sys.path.insert(0, str(_nbdir))
from pipeline.interactive import discover_conditions, make_focus_panel


def _apply_focus_nb02(cond, T):
    global FOCUS_CONDITION, FOCUS_T
    FOCUS_CONDITION = cond
    FOCUS_T = T


make_focus_panel(
    conditions = discover_conditions(_nbdir / sample_id, require="stage1_labeling.xlsx"),
    temps      = [600, 575, 550, 525, 500, 475, 450, 425, 400],
    set_focus  = _apply_focus_nb02,
    init_cond  = FOCUS_CONDITION,
    init_T     = FOCUS_T,
)


In [ ]:
import sys
import gc
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.ingest import load_ism
from pipeline.quality import (
    run_linkk, select_best_replica, compute_frequency_cutoffs, kk_summary_table,
    strip_inductive,
)
from pipeline.plots import apply_pub_style, COLOR_MAP

apply_pub_style()

sample_dir   = NOTEBOOK_DIR / sample_id
results_base = sample_dir / "Results"
all_conditions = sorted([
    d.name for d in results_base.iterdir()
    if d.is_dir() and (d / "stage1_labeling.xlsx").exists()
])

conditions = [c for c in all_conditions if c in condition_filter] if condition_filter else all_conditions[:]
if FOCUS_CONDITION is not None:
    conditions = [c for c in conditions if c == FOCUS_CONDITION]
    print(f"FOCUS_CONDITION active: {FOCUS_CONDITION}")
if SKIP_EXISTING:
    skipped    = [c for c in conditions if (results_base / c / "stage2_kk.xlsx").exists()]
    conditions = [c for c in conditions if not (results_base / c / "stage2_kk.xlsx").exists()]
    if skipped:
        print(f"SKIP_EXISTING=True: skipping {len(skipped)} already computed condition(s)")

_LABELED_RE = re.compile(r'_\d{2,4}[Cc](?:_\d+)?\.ism$', re.IGNORECASE)


def _short_cond(name: str) -> str:
    stripped = name[len(sample_id):].lstrip("_") if name.startswith(sample_id) else name
    parts = stripped.split("_")
    if parts and len(parts[0]) <= 3 and not re.match(r"^(Ar|O2|N2|H2)", parts[0], re.I):
        stripped = "_".join(parts[1:])
    parts = stripped.split("_")
    if len(parts) >= 4 and parts[-3].isdigit() and parts[-2].isdigit():
        t_hi, t_lo = parts[-3], parts[-2]
        gas = " ".join(parts[:-3])
        return f"{gas} | {t_lo}-{t_hi}C"
    return stripped


n_ism_total = 0
for c in conditions:
    xlsx_path = results_base / c / "stage1_labeling.xlsx"
    if xlsx_path.exists():
        try:
            df_tmp = pd.read_excel(xlsx_path, sheet_name="VALID")
            n_ism_total += df_tmp["file"].apply(lambda f: bool(_LABELED_RE.search(str(f)))).sum()
        except Exception:
            pass

print(f"Sample     : {sample_id}")
print(f"Conditions : {len(conditions)}")
for c in conditions:
    print(f"  {_short_cond(c)}")
print(f"Estimated  : {n_ism_total} spectra x ~20 ms = {n_ism_total * 0.020:.0f} s")

## Step 1: Batch KK (silent)

Runs Lin-KK on every (condition, T) without showing any plot. Each spectrum is
classified GREEN / YELLOW / RED based on kk_score and the fraction of points cut.
Inductive points (Z_im < 0) are stripped first.

After the table appears, look at the auto-generated `KK_OVERRIDES` suggestion in
the next cell and copy it into Step 2 if anything is flagged.

In [ ]:
# Cell A — silent batch run + classification table

def _resolve_cutoffs(condition: str, T_int: int) -> tuple:
    cond_ov = KK_OVERRIDES.get(condition, {})
    t_ov    = cond_ov.get(T_int, {})
    f_min_h = t_ov.get("f_min_hard", cond_ov.get("f_min_hard", KK_F_MIN_HARD))
    f_max_h = t_ov.get("f_max_hard", cond_ov.get("f_max_hard", KK_F_MAX_HARD))
    return f_min_h, f_max_h


def _n_in_hard_window(freq_b: np.ndarray, f_min_h, f_max_h) -> int:
    """Points within the user-defined hard limits (intentional cuts excluded from % cut)."""
    lo = f_min_h if f_min_h is not None else float(freq_b.min())
    hi = f_max_h if f_max_h is not None else float(freq_b.max())
    n  = int(((freq_b >= lo) & (freq_b <= hi)).sum())
    return n if n > 0 else len(freq_b)


def _classify_kk(kk_score: float, n_kept: int, n_total: int,
                 W_re: float | None = None, W_im: float | None = None) -> str:
    """Classify a spectrum as GREEN / YELLOW / RED.

    When ``KK_USE_W_CRITERIA`` is False, use the standard aggregate kk_score
    threshold (Schoenleber 2014). When True, use a dual-component criterion
    on W_re and W_im (Shapiro-Wilk on real and imaginary residuals
    separately). ceramic electrolyte ceramics often show W_im 0.02-0.04 below W_re because
    Z_im is small in modulus, so the aggregate kk_score is pulled below 0.97
    even for physically clean data.
    """
    frac_cut = 1.0 - (n_kept / n_total) if n_total > 0 else 1.0
    if KK_USE_W_CRITERIA and (W_re is not None) and (W_im is not None):
        if W_re >= 0.95 and W_im >= 0.93 and frac_cut <= 0.20:
            return "GREEN"
        if W_re >= 0.90 and W_im >= 0.88 and frac_cut <= 0.40:
            return "YELLOW"
        return "RED"
    if kk_score >= 0.97 and frac_cut <= 0.20:
        return "GREEN"
    if kk_score >= 0.90 and frac_cut <= 0.40:
        return "YELLOW"
    return "RED"


if FOCUS_T is not None:
    print(f"FOCUS_T = {FOCUS_T} °C → processing only T={FOCUS_T}°C across all conditions")

all_kk_data = {}
_kk_class   = {}

for condition in tqdm(conditions, desc="Conditions", unit="cond"):
    xlsx_path     = sample_dir / "Results" / condition / "stage1_labeling.xlsx"
    df_stage1_all = pd.read_excel(xlsx_path, sheet_name="VALID")
    labeled_mask  = df_stage1_all["file"].apply(lambda f: bool(_LABELED_RE.search(str(f))))
    df_stage1     = df_stage1_all[labeled_mask].copy()
    t_groups      = sorted(df_stage1["T_nominal"].dropna().unique())

    cond_data      = {}
    cond_class     = {}
    validation_dir = sample_dir / "ISM validation" / condition

    for T in t_groups:
        T_int = int(T)
        if FOCUS_T is not None and T_int != FOCUS_T:
            continue

        f_min_h, f_max_h = _resolve_cutoffs(condition, T_int)
        group_df         = df_stage1[df_stage1["T_nominal"] == T].sort_values("replica")

        records, kk_results = [], []
        for _, row in group_df.iterrows():
            ism_path = validation_dir / row["file"]
            if not ism_path.exists():
                continue
            rec = load_ism(ism_path)
            rec.T_nominal = T
            rec.T_mean    = row.get("T_mean")
            rec.pO2_mean  = row.get("pO2_mean")
            rec.replica   = row.get("replica")
            records.append(rec)

        if not records:
            continue

        for rec in records:
            freq_c, Z_re_c, Z_im_c, _ = strip_inductive(rec.freq, rec.Z_re, rec.Z_im)
            res = run_linkk(
                freq_c, Z_re_c, Z_im_c,
                c=KK_C, use_binary_M=KK_USE_BINARY_M, mu_target=KK_MU_TARGET,
                iqr_fence_factor=KK_IQR_FENCE, iqr_window=KK_IQR_WINDOW,
                f_min_hard=f_min_h, f_max_hard=f_max_h,
            )
            kk_results.append(res)

        best_idx = select_best_replica(kk_results)
        override_val = (OVERRIDES.get(condition, {}).get(T_int) or
                        OVERRIDES.get(condition, {}).get(T))
        if override_val:
            names = [r.path.name for r in records]
            if isinstance(override_val, list):
                compare_idx = [names.index(f) for f in override_val if f in names]
                if compare_idx:
                    best_idx = max(compare_idx, key=lambda i: kk_results[i]["kk_score"])
            elif override_val in names:
                best_idx = names.index(override_val)

        best     = kk_results[best_idx]
        freq_b   = best["freq"]
        n_window = _n_in_hard_window(freq_b, f_min_h, f_max_h)
        n_kept   = int(((freq_b >= best["f_min_cut"]) & (freq_b <= best["f_max_cut"])).sum())
        cls      = _classify_kk(best["kk_score"], n_kept, n_window,
                                W_re=best["W_re"], W_im=best["W_im"])

        cond_data[T_int] = {
            "records":       records,
            "kk_results":    kk_results,
            "best_idx":      best_idx,
            "selected_file": records[best_idx].path.name,
            "f_min_cut":     best["f_min_cut"],
            "f_max_cut":     best["f_max_cut"],
            "df_summary":    kk_summary_table(records, kk_results, best_idx),
        }
        cond_class[T_int] = cls

    all_kk_data[condition] = cond_data
    _kk_class[condition]   = cond_class
    gc.collect()


# Build summary DataFrame
_STATUS_COLORS = {"GREEN": "#d4edda", "YELLOW": "#fff3cd", "RED": "#f8d7da"}
_STATUS_ICONS  = {"GREEN": "✓ GREEN", "YELLOW": "⚠ YELLOW", "RED": "✗ RED"}


def _build_summary_table():
    """Rebuild the styler table from current _kk_class state. Called also by the
    widget panel when the criterion toggle changes."""
    rows = []
    for condition in conditions:
        for T_int in sorted(_kk_class.get(condition, {}).keys(), reverse=True):
            data     = all_kk_data[condition][T_int]
            best     = data["kk_results"][data["best_idx"]]
            freq_b   = best["freq"]
            f_min_h, f_max_h = _resolve_cutoffs(condition, T_int)
            n_window = _n_in_hard_window(freq_b, f_min_h, f_max_h)
            n_kept   = int(((freq_b >= best["f_min_cut"]) & (freq_b <= best["f_max_cut"])).sum())
            frac_cut_pct = round((1.0 - n_kept / n_window) * 100, 1) if n_window else 100.0
            rows.append({
                "condition":  _short_cond(condition),
                "T [°C]":     T_int,
                "file":       data["selected_file"],
                "kk_score":   round(best["kk_score"], 3),
                "W_re":       round(best["W_re"], 3),
                "W_im":       round(best["W_im"], 3),
                "f_min [Hz]": (round(best["f_min_cut"], 1)
                               if best["f_min_cut"] is not None else None),
                "f_max [Hz]": (round(best["f_max_cut"], 1)
                               if best["f_max_cut"] is not None else None),
                "% cut":      frac_cut_pct,
                "STATUS":     _STATUS_ICONS[_kk_class[condition][T_int]],
            })
    return pd.DataFrame(rows)


def _hl_status(val):
    for k, lbl in _STATUS_ICONS.items():
        if val == lbl:
            return f"background-color: {_STATUS_COLORS[k]}"
    return ""


df_summary = _build_summary_table()

styler = (
    df_summary.style
    .map(_hl_status, subset=["STATUS"])
    .format({"kk_score": "{:.3f}", "W_re": "{:.3f}", "W_im": "{:.3f}",
             "% cut": "{:.1f}"})
    .hide(axis="index")
)
display(styler)

n_green  = sum(v == "GREEN"  for d in _kk_class.values() for v in d.values())
n_yellow = sum(v == "YELLOW" for d in _kk_class.values() for v in d.values())
n_red    = sum(v == "RED"    for d in _kk_class.values() for v in d.values())
n_total  = n_green + n_yellow + n_red
t_label  = f" (FOCUS_T={FOCUS_T}°C)" if FOCUS_T is not None else ""
crit_lbl = "W-criteria (ceramic electrolyte)" if KK_USE_W_CRITERIA else "kk_score≥0.97 (strict)"
print(f"\n{n_total} spectra{t_label}: {n_green} ✓ GREEN | {n_yellow} ⚠ YELLOW | {n_red} ✗ RED  [{crit_lbl}]")
if n_yellow + n_red > 0:
    print("→ Check the suggested KK_OVERRIDES in the next cell, then run Step 2.")
else:
    print("→ All clean. Skip Step 2 and go straight to Step 3 (export).")

In [ ]:
# Cell B — suggested KK_OVERRIDES (auto-generated from flagged spectra above)

_inv = {v: k for k, v in _STATUS_ICONS.items()}

needs = {
    cond: {
        T_int: {
            "f_min_hard": round(float(
                all_kk_data[cond][T_int]["kk_results"][
                    all_kk_data[cond][T_int]["best_idx"]
                ]["f_min_cut"]), 2),
            "f_max_hard": round(float(
                all_kk_data[cond][T_int]["kk_results"][
                    all_kk_data[cond][T_int]["best_idx"]
                ]["f_max_cut"]), 2),
        }
        for T_int, cls in T_dict.items()
        if cls in ("YELLOW", "RED")
    }
    for cond, T_dict in _kk_class.items()
    if any(cls in ("YELLOW", "RED") for cls in T_dict.values())
}

if not needs:
    print("All spectra GREEN — no overrides needed. Go to Step 3.")
else:
    print("# Suggested KK_OVERRIDES — paste into Step 2 below, adjust if needed")
    print("KK_OVERRIDES = {")
    for cond, T_dict in needs.items():
        print(f'    "{cond}": {{')
        for T_int, ov in sorted(T_dict.items(), reverse=True):
            tag = _kk_class[cond][T_int]
            print(f'        {T_int}: {{"f_min_hard": {ov["f_min_hard"]}, '
                  f'"f_max_hard": {ov["f_max_hard"]}}},  # {tag}')
        print("    },")
    print("}")

    # One-click apply (no copy-paste): merge suggestions into live KK_OVERRIDES.
    try:
        import ipywidgets as _W
        from IPython.display import display as _disp
        _btn = _W.Button(description="📥 Apply suggested overrides", button_style="warning",
                         layout=_W.Layout(width="300px"),
                         tooltip="Merge the suggestions above into the in-memory KK_OVERRIDES, then re-run Step 1")
        _msg = _W.HTML()
        def _apply_kk(_b):
            n = 0
            for _c, _td in needs.items():
                KK_OVERRIDES.setdefault(_c, {}).update(_td); n += len(_td)
            _msg.value = (f"<b style='color:#b36b00'>Applied {n} override(s)</b> to KK_OVERRIDES "
                          "— re-run Step 1 (batch KK) to use them.")
        _btn.on_click(_apply_kk)
        _disp(_W.VBox([_btn, _msg]))
    except Exception as _e:
        print(f"[INFO] apply button needs ipywidgets ({_e}).")

## Step 2: Review flagged spectra

Paste the `KK_OVERRIDES` dict from Cell B below, edit values if needed, then run Cell D.
Plots appear only for spectra matching `REVIEW_THRESHOLD`. The Step 1 table updates automatically.

**Adjusting frequency margins** — use the residual plots in Cell D, not just the numbers:
- `f_min_hard`: raise it until the LF scatter disappears (cement-paste noise, electrode polarization wall)
- `f_max_hard`: lower it until the HF scatter disappears (wire inductance); the right panel zooms this region
- Typical adjustment: ±10–30% from the Cell B suggestion. Beyond that you are overfitting the window.
- kk_score > 0.90 is usually sufficient. Physical validation happens in Stage 3 via DRT vs temperature.

**About M** — `KK_USE_BINARY_M = False` (linear search) gives the same M for every replica of the
same condition, making comparisons meaningful. M is an RC-element count in the Voigt circuit, not
a peak count. Whether a peak is physical or an artifact is decided in Stage 3 by checking whether
it moves smoothly with temperature across all conditions.

**Comparing replicas at the same T** — set `OVERRIDES` in Cell C to a list of filenames:

```python
OVERRIDES = {
    "SAMPLE_ID_Ar-SCCM_O2-SCCM_Tmax_Tmin_delta": {
        400: ["sample_id_condition_TempC.ism",
              "sample_id_condition_TempC.ism"],
    }
}
```

Cell D plots all listed files, then auto-selects the highest kk_score (marked `← selected`).
Use a single string instead of a list to force a specific file without comparison.

In [ ]:
# Quick KK tuning panel — re-run Lin-KK for one (condition, T) with custom f_min / f_max.
# Updates KK_OVERRIDES in memory; re-run Step 3 (export) afterwards to persist.
try:
    import ipywidgets as W
    from IPython.display import display as _display, clear_output as _clear, HTML as _HTML
    _HAS_WIDGETS_NB02 = True
except Exception as _exc:
    print(f"[INFO] ipywidgets not installed ({_exc}); KK tuning panel disabled.")
    _HAS_WIDGETS_NB02 = False


def _reclassify_all() -> None:
    """Recompute _kk_class based on current KK_USE_W_CRITERIA flag."""
    global _kk_class
    for cond, td in all_kk_data.items():
        for T_int, data in td.items():
            best     = data["kk_results"][data["best_idx"]]
            freq_b   = best["freq"]
            f_min_h, f_max_h = _resolve_cutoffs(cond, T_int)
            n_window = _n_in_hard_window(freq_b, f_min_h, f_max_h)
            n_kept   = int(((freq_b >= best["f_min_cut"]) & (freq_b <= best["f_max_cut"])).sum())
            cls      = _classify_kk(best["kk_score"], n_kept, n_window,
                                    W_re=best["W_re"], W_im=best["W_im"])
            _kk_class.setdefault(cond, {})[T_int] = cls


def _count_status() -> tuple[int, int, int]:
    g = sum(v == "GREEN"  for d in _kk_class.values() for v in d.values())
    y = sum(v == "YELLOW" for d in _kk_class.values() for v in d.values())
    r = sum(v == "RED"    for d in _kk_class.values() for v in d.values())
    return g, y, r


def _status_chips_html() -> str:
    g, y, r = _count_status()
    crit = "W-criteria (ceramic electrolyte)" if KK_USE_W_CRITERIA else "kk_score≥0.97 (strict)"
    return (f"<div style='font-size:13px; padding:4px 0'>"
            f"<span style='background:#d4edda; padding:3px 8px; border-radius:4px'>🟢 {g}</span>&nbsp; "
            f"<span style='background:#fff3cd; padding:3px 8px; border-radius:4px'>🟡 {y}</span>&nbsp; "
            f"<span style='background:#f8d7da; padding:3px 8px; border-radius:4px'>🔴 {r}</span>&nbsp;&nbsp; "
            f"<i>criterion: {crit}</i></div>")


def _retest_kk(condition: str, T_int: int, f_min: float | None, f_max: float | None,
               fence: float | None = None, window: int | None = None) -> None:
    """Re-run Lin-KK on the best replica for (condition, T) with custom cutoffs.

    Persists the choice in ``KK_OVERRIDES`` (in memory) and shows the residual plot.
    """
    data = all_kk_data.get(condition, {}).get(T_int)
    if data is None:
        print(f"[WARN] no KK data for {condition} T={T_int} — run Step 1 first.")
        return
    rec = data["records"][data["best_idx"]]

    if f_min is not None or f_max is not None:
        KK_OVERRIDES.setdefault(condition, {}).setdefault(T_int, {})
        if f_min is not None: KK_OVERRIDES[condition][T_int]["f_min_hard"] = f_min
        if f_max is not None: KK_OVERRIDES[condition][T_int]["f_max_hard"] = f_max

    freq_c, Z_re_c, Z_im_c, n_ind = strip_inductive(rec.freq, rec.Z_re, rec.Z_im)
    res = run_linkk(
        freq_c, Z_re_c, Z_im_c,
        c=KK_C, use_binary_M=KK_USE_BINARY_M, mu_target=KK_MU_TARGET,
        iqr_fence_factor=(fence if fence is not None else KK_IQR_FENCE),
        iqr_window=(window if window is not None else KK_IQR_WINDOW),
        f_min_hard=f_min, f_max_hard=f_max,
    )
    freq_b = res["freq"]
    n_win  = _n_in_hard_window(freq_b, f_min, f_max)
    n_kept = int(((freq_b >= res["f_min_cut"]) & (freq_b <= res["f_max_cut"])).sum())
    cls    = _classify_kk(res["kk_score"], n_kept, n_win, W_re=res["W_re"], W_im=res["W_im"])
    print(f"  kk={res['kk_score']:.4f}  W_re={res['W_re']:.3f}  W_im={res['W_im']:.3f}  "
          f"M={res['M']}  μ={res['mu']:.2f}  [{cls}]  "
          f"f_min_cut={res['f_min_cut']:.1f} Hz  f_max_cut={res['f_max_cut']:.1f} Hz")

    fig, ax = plt.subplots(figsize=(8, 3.2))
    color = COLOR_MAP.get(T_int, "#555555")
    fence_pc = res["cutoff_fence"] * 100
    ax.semilogx(freq_b, res["res_re"]*100, "o-", ms=2.5, lw=0.8, color=color, label="Re")
    ax.semilogx(freq_b, res["res_im"]*100, "o-", ms=2.5, lw=0.8, color="darkorange", label="Im")
    ax.axhline( fence_pc, color="steelblue", ls="--", lw=0.9, label=f"±{fence_pc:.1f}%")
    ax.axhline(-fence_pc, color="steelblue", ls="--", lw=0.9)
    ax.axhline(0, color="k", lw=0.4)
    if res["f_min_cut"]: ax.axvline(res["f_min_cut"], color="grey", ls="--", lw=1.0)
    if res["f_max_cut"]: ax.axvline(res["f_max_cut"], color="grey", ls="-.", lw=1.0)
    ax.set_xlabel("Frequency [Hz]"); ax.set_ylabel("Residual / |Z| [%]")
    ax.set_title(f"{condition}  |  T={T_int}°C  |  panel re-test  [{cls}]", fontsize=9)
    ax.grid(True, which="both", ls=":", alpha=0.3)
    ax.legend(fontsize=7, frameon=False, loc="upper left")
    plt.tight_layout()
    plt.show()


# KK presets — one-click parameter sets
_KK_PRESETS = {
    "Conservative (publication)": dict(
        KK_C=0.85, KK_IQR_FENCE=2.0, KK_IQR_WINDOW=5,
        KK_F_MIN_HARD=30, KK_USE_W_CRITERIA=False),
    "Standard (current optimum)": dict(
        KK_C=0.76, KK_IQR_FENCE=0.5, KK_IQR_WINDOW=10,
        KK_F_MIN_HARD=50, KK_USE_W_CRITERIA=False),
    "Permissive (ceramic electrolyte-aware)": dict(
        KK_C=0.76, KK_IQR_FENCE=0.5, KK_IQR_WINDOW=10,
        KK_F_MIN_HARD=50, KK_USE_W_CRITERIA=True),
    "RelaxIS defaults": dict(
        KK_C=0.85, KK_IQR_FENCE=2.0, KK_IQR_WINDOW=5,
        KK_F_MIN_HARD=None, KK_USE_W_CRITERIA=False),
}


if _HAS_WIDGETS_NB02 and all_kk_data:
    _conds_nb02 = [c for c, d in all_kk_data.items() if d]
    if _conds_nb02:
        _c0_2 = _conds_nb02[0]
        _Ts_2 = sorted(all_kk_data[_c0_2].keys(), reverse=True)
        _T0_2 = _Ts_2[0] if _Ts_2 else 600

        wc  = W.Dropdown(options=_conds_nb02, value=_c0_2, description="Cond:",
                         layout=W.Layout(width="420px"))
        wT  = W.Dropdown(options=_Ts_2, value=_T0_2, description="T [°C]:",
                         layout=W.Layout(width="200px"))
        wmn = W.FloatLogSlider(value=KK_F_MIN_HARD or 30, base=10, min=0, max=6, step=0.1,
                               description="f_min Hz", readout_format=".1f",
                               continuous_update=False, layout=W.Layout(width="360px"))
        wmx = W.FloatLogSlider(value=KK_F_MAX_HARD or 1e6, base=10, min=2, max=8, step=0.1,
                               description="f_max Hz", readout_format=".1e",
                               continuous_update=False, layout=W.Layout(width="360px"))
        # The edge-trim CONSTRAINT: lower fence = stricter cut at noisy edges.
        wfence = W.FloatSlider(value=KK_IQR_FENCE, min=0.1, max=3.0, step=0.1,
                               description="fence", readout_format=".1f",
                               continuous_update=False, layout=W.Layout(width="360px"),
                               tooltip="IQR fence multiplier - LOWER = stricter edge cut")
        wwin = W.IntSlider(value=KK_IQR_WINDOW, min=3, max=20, step=1,
                           description="window", continuous_update=False,
                           layout=W.Layout(width="300px"),
                           tooltip="consecutive clean points to confirm the cut edge")
        w_crit = W.ToggleButton(
            value=bool(KK_USE_W_CRITERIA),
            description=("ceramic electrolyte W-criteria" if KK_USE_W_CRITERIA else "strict kk≥0.97"),
            button_style="info", layout=W.Layout(width="170px"),
            tooltip="Click to switch classification criterion (live re-colours table)")
        wgo = W.Button(description="↻ Re-test KK", button_style="primary",
                       layout=W.Layout(width="160px"))

        # Preset dropdown (click "Apply preset" to load values)
        wpreset = W.Dropdown(options=list(_KK_PRESETS), value="Standard (current optimum)",
                             description="Preset:", layout=W.Layout(width="380px"))
        w_apply = W.Button(description="📥 Apply preset", button_style="warning",
                           layout=W.Layout(width="160px"))

        chips = W.HTML(value=_status_chips_html())
        out2  = W.Output()

        def _load_state_into_widgets(*_):
            """Update widgets to show stored override for current (cond, T)."""
            cond, T = wc.value, int(wT.value)
            ov = KK_OVERRIDES.get(cond, {})
            t_ov = ov.get(T, {}) if isinstance(ov.get(T), dict) else {}
            fmin = t_ov.get("f_min_hard",
                            ov.get("f_min_hard") if not isinstance(ov.get("f_min_hard"), dict) else None)
            fmax = t_ov.get("f_max_hard",
                            ov.get("f_max_hard") if not isinstance(ov.get("f_max_hard"), dict) else None)
            wmn.value = float(fmin) if fmin is not None else (KK_F_MIN_HARD or 30)
            wmx.value = float(fmax) if fmax is not None else (KK_F_MAX_HARD or 1e6)

        def _refresh_T_nb02(*_):
            ts = sorted(all_kk_data.get(wc.value, {}).keys(), reverse=True)
            wT.options = ts
            if ts and wT.value not in ts:
                wT.value = ts[0]
            _load_state_into_widgets()

        wc.observe(_refresh_T_nb02, names="value")
        wT.observe(_load_state_into_widgets, names="value")

        def _on_crit_toggle(change):
            global KK_USE_W_CRITERIA
            KK_USE_W_CRITERIA = bool(change["new"])
            w_crit.description = "ceramic electrolyte W-criteria" if KK_USE_W_CRITERIA else "strict kk≥0.97"
            _reclassify_all()
            chips.value = _status_chips_html()
            with out2:
                _clear(wait=True)
                g, y, r = _count_status()
                print(f"Criterion switched to "
                      f"{'ceramic electrolyte W-criteria (W_re≥0.95 AND W_im≥0.93)' if KK_USE_W_CRITERIA else 'strict kk≥0.97'}")
                print(f"  {g} GREEN | {y} YELLOW | {r} RED")
        w_crit.observe(_on_crit_toggle, names="value")

        def _on_retest(_btn=None):
            global KK_IQR_FENCE, KK_IQR_WINDOW
            KK_IQR_FENCE  = float(wfence.value)
            KK_IQR_WINDOW = int(wwin.value)
            with out2:
                _clear(wait=True)
                _retest_kk(wc.value, int(wT.value),
                           float(wmn.value) if wmn.value > 0 else None,
                           float(wmx.value) if wmx.value > 0 else None,
                           fence=float(wfence.value), window=int(wwin.value))
            _reclassify_all()
            chips.value = _status_chips_html()
        wgo.on_click(_on_retest)
        # Live: re-test on slider release (continuous_update=False keeps it from firing mid-drag)
        for _w in (wfence, wwin, wmn, wmx):
            _w.observe(lambda ch: _on_retest(), names="value")

        def _on_apply_preset(_btn):
            global KK_C, KK_IQR_FENCE, KK_IQR_WINDOW, KK_F_MIN_HARD, KK_USE_W_CRITERIA
            p = _KK_PRESETS[wpreset.value]
            KK_C            = p["KK_C"]
            KK_IQR_FENCE    = p["KK_IQR_FENCE"]
            KK_IQR_WINDOW   = p["KK_IQR_WINDOW"]
            KK_F_MIN_HARD   = p["KK_F_MIN_HARD"]
            KK_USE_W_CRITERIA = p["KK_USE_W_CRITERIA"]
            w_crit.value = bool(KK_USE_W_CRITERIA)  # triggers reclassify + chips
            wmn.value    = float(KK_F_MIN_HARD) if KK_F_MIN_HARD is not None else 30.0
            wfence.value = float(KK_IQR_FENCE)
            wwin.value   = int(KK_IQR_WINDOW)
            with out2:
                _clear(wait=True)
                print(f"Loaded preset '{wpreset.value}':")
                for k, v in p.items():
                    print(f"  {k} = {v}")
                print("Re-run Step 1 (cell above) to apply to the full batch.")
        w_apply.on_click(_on_apply_preset)

        _load_state_into_widgets()
        _display(W.VBox([
            W.HBox([wc, wT, w_crit]),
            W.HBox([wmn, wmx]),
            W.HBox([wfence, wwin, wgo]),
            W.HBox([wpreset, w_apply]),
            chips, out2,
        ]))
elif not all_kk_data:
    print("[INFO] No KK data — run Step 1 first.")

In [ ]:
# Cell C — review controls (edit then run Cell D below)

# "yellow" → review YELLOW + RED   |   "red" → review only RED
REVIEW_THRESHOLD = "yellow"


# Suggested KK_OVERRIDES — paste into Step 2 below, adjust if needed
KK_OVERRIDES = {}

OVERRIDES = {
    # Single file (force one replica):
    #   "condition": {400: "filename_400C_1.ism"}
    #
    # Multiple files (comparison mode — Cell D plots all, auto-selects best kk_score):
    #   "condition": {400: ["filename_400C.ism", "filename_400C_1.ism", "filename_400C_2.ism"]}
}

In [ ]:
# Cell D — targeted review (residual plots only for flagged spectra)

_target_map = {"yellow": {"YELLOW", "RED"}, "red": {"RED"}}
_target = _target_map.get(REVIEW_THRESHOLD, {"YELLOW", "RED"})

n_reviewed = 0
for condition in conditions:
    cond_class_local = _kk_class.get(condition, {})
    for T_int in sorted(cond_class_local.keys(), reverse=True):
        if cond_class_local[T_int] not in _target:
            continue

        n_reviewed     += 1
        f_min_h, f_max_h = _resolve_cutoffs(condition, T_int)
        data    = all_kk_data[condition][T_int]
        records = data["records"]

        new_kk = []
        for rec in records:
            freq_c, Z_re_c, Z_im_c, n_ind = strip_inductive(rec.freq, rec.Z_re, rec.Z_im)
            res = run_linkk(
                freq_c, Z_re_c, Z_im_c,
                c=KK_C, use_binary_M=KK_USE_BINARY_M, mu_target=KK_MU_TARGET,
                iqr_fence_factor=KK_IQR_FENCE, iqr_window=KK_IQR_WINDOW,
                f_min_hard=f_min_h, f_max_hard=f_max_h,
            )
            res["_n_inductive"] = n_ind
            new_kk.append(res)

        best_idx = select_best_replica(new_kk)
        names = [r.path.name for r in records]

        override_val = (OVERRIDES.get(condition, {}).get(T_int) or
                        OVERRIDES.get(condition, {}).get(float(T_int)))

        if isinstance(override_val, list):
            # Comparison mode: show a plot for each listed file, then pick best kk_score among them
            compare_idx = [names.index(f) for f in override_val if f in names]
            missing     = [f for f in override_val if f not in names]
            if missing:
                print(f"  [T={T_int}] OVERRIDES — files not found: {missing}")
            if not compare_idx:
                compare_idx = [best_idx]
            plot_indices = compare_idx
            best_idx = max(compare_idx, key=lambda i: new_kk[i]["kk_score"])
            print(f"  [T={T_int}] Comparison mode: {len(plot_indices)} files — "
                  f"auto-selecting best kk_score → {names[best_idx]}\n")
        elif override_val and override_val in names:
            best_idx     = names.index(override_val)
            plot_indices = [best_idx]
        else:
            plot_indices = [best_idx]

        for _pidx in plot_indices:
            rec_p  = records[_pidx]
            res_p  = new_kk[_pidx]
            freq_b = res_p["freq"]
            n_window = _n_in_hard_window(freq_b, f_min_h, f_max_h)
            n_kept   = int(((freq_b >= res_p["f_min_cut"]) & (freq_b <= res_p["f_max_cut"])).sum())
            cls_p    = _classify_kk(res_p["kk_score"], n_kept, n_window,
                                    W_re=res_p["W_re"], W_im=res_p["W_im"])

            _sel_tag  = "  ← selected" if (_pidx == best_idx and len(plot_indices) > 1) else ""
            _color    = COLOR_MAP.get(T_int, "#555555")
            _fence_pc = res_p["cutoff_fence"] * 100
            fmin_c    = res_p["f_min_cut"]
            fmax_c    = res_p["f_max_cut"]

            fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
            fig.suptitle(
                f"{_short_cond(condition)}  |  T = {T_int} °C  |  {rec_p.path.name}{_sel_tag}\n"
                f"kk = {res_p['kk_score']:.4f}   M = {res_p['M']}   mu = {res_p['mu']:.2f}   "
                f"[{cls_p}]   stripped {res_p['_n_inductive']} inductive pts",
                fontsize=9, y=1.02,
            )

            f_hf_thresh = np.percentile(freq_b, 80)

            for ax, zoom in [(axes[0], False), (axes[1], True)]:
                for res_arr, lbl, col in [
                    (res_p["res_re"] * 100, "Re", _color),
                    (res_p["res_im"] * 100, "Im", "darkorange"),
                ]:
                    ax.semilogx(freq_b, res_arr, "o-", ms=2.5, lw=0.8,
                                color=col, label=lbl, alpha=0.85)
                ax.axhline( _fence_pc, color="steelblue", ls="--", lw=0.9,
                            label=f"+/-{_fence_pc:.1f}%")
                ax.axhline(-_fence_pc, color="steelblue", ls="--", lw=0.9)
                ax.axhline(0, color="k", lw=0.4)
                if fmin_c is not None:
                    ax.axvline(fmin_c, color="grey", ls="--", lw=1.0,
                               label=f"f_min={fmin_c:.2g} Hz")
                if fmax_c is not None:
                    ax.axvline(fmax_c, color="grey", ls="-.", lw=1.0,
                               label=f"f_max={fmax_c:.2g} Hz")
                ax.set_xlabel("Frequency [Hz]")
                ax.set_ylabel("Residual / |Z| [%]")
                ax.grid(True, which="both", ls=":", alpha=0.3)
                if zoom:
                    ax.set_xlim(f_hf_thresh, freq_b.max() * 1.05)
                    ax.set_title("HF zoom (top 20% freq)", fontsize=9)
                else:
                    ax.set_title("Full range", fontsize=9)
                    ax.legend(fontsize=7, frameon=False, loc="upper left")
            plt.tight_layout()
            plt.show()

            sym = {"GREEN": "✓", "YELLOW": "⚠", "RED": "✗"}[cls_p]
            print(f"  {sym} {cls_p}  |  kk={res_p['kk_score']:.4f}  |  "
                  f"f_min={fmin_c:.2g} Hz  |  f_max={fmax_c:.2g} Hz  |  {rec_p.path.name}")

        # Finalise with the selected best_idx (best kk among comparison list, or override, or auto)
        final_res  = new_kk[best_idx]
        final_freq = final_res["freq"]
        n_window_f = _n_in_hard_window(final_freq, f_min_h, f_max_h)
        final_kept = int(((final_freq >= final_res["f_min_cut"]) &
                          (final_freq <= final_res["f_max_cut"])).sum())
        new_cls = _classify_kk(final_res["kk_score"], final_kept, n_window_f,
                               W_re=final_res["W_re"], W_im=final_res["W_im"])

        if len(plot_indices) > 1:
            sym = {"GREEN": "✓", "YELLOW": "⚠", "RED": "✗"}[new_cls]
            print(f"\n  → Export: {records[best_idx].path.name}  {sym} {new_cls}\n")

        all_kk_data[condition][T_int] = {
            "records":       records,
            "kk_results":    new_kk,
            "best_idx":      best_idx,
            "selected_file": records[best_idx].path.name,
            "f_min_cut":     final_res["f_min_cut"],
            "f_max_cut":     final_res["f_max_cut"],
            "df_summary":    kk_summary_table(records, new_kk, best_idx),
        }
        _kk_class[condition][T_int] = new_cls

if n_reviewed == 0:
    print("Nothing to review — all spectra already pass the threshold.")
else:
    print(f"\nReviewed {n_reviewed} spectrum/spectra. Edit Cell C and re-run if needed.")
    print("→ When happy, go to Step 3 (export).")

## Step 3: Selection summary and export

Compact per-condition table of the selected replicas and writes `stage2_kk.xlsx`
(All + Selected sheets) consumed by Stage 3.

When `FOCUS_T` is set, the export **merges** into the existing file — rows for other
temperatures are preserved. `FOCUS_T = None` → full overwrite (safe for first run).

In [ ]:
# Summary Styler table per condition
from pipeline.utils import merge_sheet_by_T, build_metadata_sheet

for condition, cond_data in all_kk_data.items():
    if not cond_data:
        continue
    print(f"\nCondition: {condition}")

    rows = []
    for T_int in sorted(cond_data.keys(), reverse=True):
        data = cond_data[T_int]
        best = data["kk_results"][data["best_idx"]]
        rec  = data["records"][data["best_idx"]]

        if   best["pass_re"] and best["pass_im"]:     kk_lbl = "✓ PASS"
        elif best["pass_re"] and not best["pass_im"]: kk_lbl = "✗ Re only"
        elif best["pass_im"] and not best["pass_re"]: kk_lbl = "✗ Im only"
        else:                                         kk_lbl = "✗ fail"

        rows.append({
            "T [°C]":    T_int,
            "pO₂ [bar]": round(float(rec.pO2_mean or 0), 4),
            "file":      data["selected_file"],
            "kk_score":  round(best["kk_score"], 3),
            "W_re":      round(best["W_re"], 3),
            "W_im":      round(best["W_im"], 3),
            "KK":        kk_lbl,
            "f_min [Hz]": (round(data["f_min_cut"], 1)
                           if data["f_min_cut"] is not None else "—"),
            "f_max [Hz]": (round(data["f_max_cut"], 1)
                           if data["f_max_cut"] is not None else "—"),
            "★": "★",
        })

    df_disp = pd.DataFrame(rows)

    def _highlight_kk(val):
        return "background-color: #fff3cd" if "✗" in str(val) else ""

    styler = (
        df_disp.style
        .map(_highlight_kk, subset=["KK"])
        .format({"kk_score": "{:.3f}", "W_re": "{:.3f}", "W_im": "{:.3f}"})
        .set_caption(condition.replace("_", " "))
        .set_table_styles([{
            "selector": "caption",
            "props": "font-size: 11px; font-weight: bold; text-align: left;"
        }])
        .hide(axis="index")
    )
    display(styler)

# Build Metadata DataFrame (Lin-KK fixed parameters — applied to all conditions)
df_meta = build_metadata_sheet(
    sample_id  = sample_id,
    stage_name = "stage2_kk",
    params = {
        "KK_C":              KK_C,
        "KK_MU_TARGET":      KK_MU_TARGET,
        "KK_USE_BINARY_M":   KK_USE_BINARY_M,
        "KK_IQR_FENCE":      KK_IQR_FENCE,
        "KK_IQR_WINDOW":     KK_IQR_WINDOW,
        "KK_F_MIN_HARD":     KK_F_MIN_HARD,
        "KK_F_MAX_HARD":     KK_F_MAX_HARD,
        "acceptance_GREEN":  "kk_score >= 0.97 AND frac_cut <= 0.20",
        "acceptance_YELLOW": "kk_score >= 0.90 AND frac_cut <= 0.40",
        "reference":         "Schoenleber et al., Electrochim. Acta 131 (2014); relaXIS manual v1.31",
    },
)

# Export stage2_kk.xlsx (merge-aware when FOCUS_T is set)
print("\nExporting...")
for condition, cond_data in all_kk_data.items():
    if not cond_data:
        print(f"  [{condition}] No data — skipped.")
        continue

    results_dir = sample_dir / "Results" / condition
    results_dir.mkdir(parents=True, exist_ok=True)
    rows_all, rows_sel = [], []

    for T_int, data in sorted(cond_data.items()):
        for i, (rec, res) in enumerate(zip(data["records"], data["kk_results"])):
            f_min, f_max = compute_frequency_cutoffs(res)
            is_sel = i == data["best_idx"]
            row = {
                "condition": condition, "file": rec.path.name,
                "full_path": str(rec.path), "T_nominal": T_int,
                "T_mean":    round(rec.T_mean, 2) if rec.T_mean is not None else None,
                "pO2_mean":  rec.pO2_mean, "replica": rec.replica,
                "kk_score":  round(res["kk_score"], 4),
                "W_re":      round(res["W_re"], 4),
                "W_im":      round(res["W_im"], 4),
                "pass_re":   res["pass_re"],
                "pass_im":   res["pass_im"],
                "mu":        round(res.get("mu", float("nan")), 3),
                "M":         res.get("M"),
                "max_res_re":    round(float(np.abs(res["res_re"]).max()), 4),
                "max_res_im":    round(float(np.abs(res["res_im"]).max()), 4),
                "cutoff_fence":  round(res.get("cutoff_fence", float("nan")), 4),
                "f_min_cut": f_min, "f_max_cut": f_max, "selected": is_sel,
            }
            rows_all.append(row)
            if is_sel:
                rows_sel.append(row)

    xlsx_path = results_dir / "stage2_kk.xlsx"

    df_all = merge_sheet_by_T(xlsx_path, "All",      pd.DataFrame(rows_all), FOCUS_T)
    df_sel = merge_sheet_by_T(xlsx_path, "Selected", pd.DataFrame(rows_sel), FOCUS_T)
    _export_mode = f"merged T={FOCUS_T}°C" if (FOCUS_T is not None and xlsx_path.exists()) else "full overwrite"

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df_all.to_excel(writer,  sheet_name="All",      index=False)
        df_sel.to_excel(writer,  sheet_name="Selected", index=False)
        df_meta.to_excel(writer, sheet_name="Metadata", index=False)

    print(f"  [{condition}]  {len(df_sel)} selected → {xlsx_path.relative_to(NOTEBOOK_DIR)}  [{_export_mode}]")

print("\nExport complete.")
print("→ Next: 03_drt_zarc.ipynb")